# CommercePulse — Segmentação de Clientes por RFM

Este notebook aplica a **análise RFM (Recency, Frequency, Monetary)** para segmentar
os clientes do e-commerce brasileiro Olist, identificando perfis de comportamento de compra
e oportunidades de retenção.

---

### O que é RFM?

| Dimensão | Significado | Quanto maior, melhor? |
|----------|-------------|-----|
| **Recency** | Quantos dias desde a última compra | Não (menor = mais recente) |
| **Frequency** | Quantas compras o cliente fez | Sim |
| **Monetary** | Quanto o cliente gastou no total | Sim |

In [1]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from pathlib import Path

BASE_DIR = Path().resolve().parent
PROCESSED_DIR = BASE_DIR / "data" / "processed"

df = pd.read_csv(PROCESSED_DIR / "commercepulse_orders_items.csv")
df["order_purchase_timestamp"] = pd.to_datetime(df["order_purchase_timestamp"], errors="coerce")

# Apenas pedidos entregues
df_delivered = df[df["order_status"] == "delivered"].copy()

print(f"Itens entregues: {len(df_delivered):,}")
print(f"Clientes únicos: {df_delivered['customer_unique_id'].nunique():,}")

Itens entregues: 110,197
Clientes únicos: 93,358


## 1. Cálculo das Métricas RFM

In [2]:
reference_date = df_delivered["order_purchase_timestamp"].max() + pd.Timedelta(days=1)
print(f"Data de referência: {reference_date.date()}")

# Receita por pedido (evita duplicação por item)
order_revenue = df_delivered.groupby(["customer_unique_id", "order_id"]).agg(
    order_revenue=("item_revenue", "sum"),
    order_date=("order_purchase_timestamp", "first"),
).reset_index()

rfm = order_revenue.groupby("customer_unique_id").agg(
    recency=("order_date", lambda x: (reference_date - x.max()).days),
    frequency=("order_id", "nunique"),
    monetary=("order_revenue", "sum"),
).reset_index()

print(f"\nClientes na tabela RFM: {len(rfm):,}")
rfm.describe()

Data de referência: 2018-08-30



Clientes na tabela RFM: 93,358


,recency,frequency,monetary
count,93358.000000,93358.000000,93358.000000
mean,237.941773,1.033420,141.621480
std,152.591453,0.209097,215.694014
min,1.000000,1.000000,0.850000
25%,114.000000,1.000000,47.650000
50%,219.000000,1.000000,89.730000
75%,346.000000,1.000000,154.737500
max,714.000000,15.000000,13440.000000


## 2. Distribuições de R, F e M

In [3]:
fig = make_subplots(rows=1, cols=3, subplot_titles=["Recency (dias)", "Frequency (compras)", "Monetary (R$)"])

fig.add_trace(go.Histogram(x=rfm["recency"], nbinsx=50, marker_color="#636EFA", name="Recency"), row=1, col=1)
fig.add_trace(go.Histogram(x=rfm["frequency"], nbinsx=20, marker_color="#00CC96", name="Frequency"), row=1, col=2)
fig.add_trace(go.Histogram(x=rfm["monetary"], nbinsx=50, marker_color="#FFA15A", name="Monetary"), row=1, col=3)

fig.update_layout(template="plotly_dark", height=400, showlegend=False, title="Distribuição das Métricas RFM")
fig.show()

## 3. Atribuição de Scores (Quintis)

In [4]:
# Percentis preservam empates: clientes com o mesmo valor recebem o mesmo score
def percentile_score(series, higher_is_better=True):
    score = np.ceil(series.rank(method="average", pct=True) * 5).clip(1, 5).astype(int)
    return score if higher_is_better else 6 - score

rfm["R_score"] = percentile_score(rfm["recency"], higher_is_better=False)

# Frequência usa faixas reais; não separa arbitrariamente quem comprou uma vez
rfm["F_score"] = np.select(
    [rfm["frequency"] <= 1, rfm["frequency"] == 2, rfm["frequency"] == 3, rfm["frequency"] == 4],
    [1, 2, 3, 4], default=5,
).astype(int)

rfm["M_score"] = percentile_score(rfm["monetary"])

rfm["RFM_score"] = rfm["R_score"] + rfm["F_score"] + rfm["M_score"]

print("Score RFM combinado (3-15):")
print(rfm["RFM_score"].describe())

Score RFM combinado (3-15):
count    93358.000000
mean         7.038251
std          2.045443
min          3.000000
25%          6.000000
50%          7.000000
75%          8.000000
max         15.000000
Name: RFM_score, dtype: float64


## 4. Segmentação de Clientes

In [5]:
def assign_rfm_segment(row):
    r, f, m = row["R_score"], row["F_score"], row["M_score"]
    if r >= 4 and f >= 4 and m >= 4:
        return "Champions"
    elif r >= 3 and f >= 3 and m >= 3:
        return "Loyal Customers"
    elif r >= 4 and f <= 2 and m <= 2:
        return "New Customers"
    elif r >= 3 and f >= 1 and m >= 2:
        return "Potential Loyalists"
    elif r <= 2 and f >= 3 and m >= 3:
        return "At Risk"
    elif r <= 2 and f >= 1 and m >= 1:
        return "Hibernating"
    else:
        return "Others"

rfm["segment"] = rfm.apply(assign_rfm_segment, axis=1)

print("Distribuição por segmento:")
print(rfm["segment"].value_counts())

Distribuição por segmento:
segment
Potential Loyalists    37577
Hibernating            37154
New Customers          14605
Others                  3803
Loyal Customers          124
At Risk                   62
Champions                 33
Name: count, dtype: int64


## 5. Visualização dos Segmentos

In [6]:
SEGMENT_COLORS = {
    "Champions": "#636EFA",
    "Loyal Customers": "#00CC96",
    "Potential Loyalists": "#AB63FA",
    "New Customers": "#FFA15A",
    "At Risk": "#EF553B",
    "Hibernating": "#FF6692",
    "Others": "#B6E880",
}

seg_counts = rfm["segment"].value_counts().reset_index()
seg_counts.columns = ["Segmento", "Clientes"]

fig = px.pie(
    seg_counts,
    names="Segmento",
    values="Clientes",
    color="Segmento",
    color_discrete_map=SEGMENT_COLORS,
    hole=0.45,
    template="plotly_dark",
    height=500,
    title="Proporção de Clientes por Segmento RFM",
)
fig.update_traces(textposition="outside", textinfo="label+percent")
fig.show()

## 6. Métricas Médias por Segmento

In [7]:
seg_metrics = rfm.groupby("segment").agg(
    clientes=("customer_unique_id", "count"),
    recency_media=("recency", "mean"),
    frequency_media=("frequency", "mean"),
    monetary_media=("monetary", "mean"),
    rfm_score_medio=("RFM_score", "mean"),
).reset_index().sort_values("monetary_media", ascending=False)

seg_metrics

,segment,clientes,recency_media,frequency_media,monetary_media,rfm_score_medio
1,Champions,33,85.969697,4.939394,726.778788,14.000000
3,Loyal Customers,124,128.362903,3.137097,393.364194,11.758065
0,At Risk,62,373.790323,3.080645,386.297097,9.370968
6,Potential Loyalists,37577,142.559491,1.040929,190.915904,8.745350
2,Hibernating,37154,395.038246,1.025246,142.107102,5.502638
4,New Customers,14605,89.370558,1.007737,39.373724,7.017117
5,Others,3803,218.872732,1.001841,25.200999,5.001841


In [8]:
fig = make_subplots(rows=1, cols=3, subplot_titles=["Recência Média", "Frequência Média", "Monetary Médio"])

for i, (col, color) in enumerate([("recency_media", "#636EFA"), ("frequency_media", "#00CC96"), ("monetary_media", "#FFA15A")]):
    sorted_data = seg_metrics.sort_values(col)
    fig.add_trace(
        go.Bar(x=sorted_data[col], y=sorted_data["segment"], orientation="h", marker_color=color, name=col),
        row=1, col=i + 1,
    )

fig.update_layout(template="plotly_dark", height=450, showlegend=False, title="Comparação de Métricas por Segmento")
fig.show()

## 7. Scatter 3D: R × F × M por Segmento

In [9]:
rfm_sample = rfm.sample(n=min(5000, len(rfm)), random_state=42)

fig = px.scatter_3d(
    rfm_sample,
    x="recency",
    y="frequency",
    z="monetary",
    color="segment",
    color_discrete_map=SEGMENT_COLORS,
    opacity=0.6,
    template="plotly_dark",
    height=650,
    labels={"recency": "Recência (dias)", "frequency": "Frequência", "monetary": "Valor (R$)", "segment": "Segmento"},
    title="Mapa 3D de Segmentos RFM",
)
fig.show()

## 8. Distribuição de Frequência de Compras

In [10]:
freq_dist = rfm["frequency"].value_counts().sort_index().reset_index()
freq_dist.columns = ["Compras", "Clientes"]

fig = px.bar(
    freq_dist.head(10),
    x="Compras",
    y="Clientes",
    color_discrete_sequence=["#636EFA"],
    template="plotly_dark",
    height=400,
    text="Clientes",
    title="Distribuição da Frequência de Compras",
)
fig.update_traces(texttemplate="%{text:,}", textposition="outside")
fig.show()

pct_single = (rfm["frequency"] == 1).mean() * 100
print(f"\n{pct_single:.1f}% dos clientes fizeram apenas 1 compra.")
print("Isso indica uma grande oportunidade de retenção e aumento de lifetime value.")


97.0% dos clientes fizeram apenas 1 compra.
Isso indica uma grande oportunidade de retenção e aumento de lifetime value.


## 9. Conclusões e Recomendações

### Principais achados:

1. **~97% de compra única:** A vasta maioria dos clientes fez apenas 1 compra. Isso é típico de marketplaces, mas representa uma **enorme oportunidade** de retenção.

2. **Champions são poucos mas valiosos:** Representam uma fatia menor dos clientes, mas com ticket significativamente mais alto.

3. **Clientes At Risk e Hibernating:** Juntos, representam uma parcela significativa — são clientes que já compraram e podem ser reativados.

### Recomendações por Segmento:

| Segmento | Ação Recomendada |
|----------|------------------|
| **Champions** | Programa de fidelidade VIP, acesso antecipado |
| **Loyal Customers** | Cross-sell, recompensas por indicação |
| **New Customers** | Cupom de segunda compra (7-14 dias após entrega) |
| **Potential Loyalists** | Ofertas personalizadas, programa de pontos |
| **At Risk** | Campanha de retenção urgente, descontos agressivos |
| **Hibernating** | Campanha de reativação com desconto e frete grátis |